# SC203 DoSSR — clean Colab L4 smoke run

Use this notebook in Google Colab. In **Runtime → Change runtime type**, select **L4 GPU** and turn on **High-RAM** if available, then save. High-RAM is system memory; it does not increase GPU VRAM.

This notebook runs a single RealSR image with FP16 and stops after verifying the output. It does not enable CPU condition-encoder offload or start the full 200-image baseline. FP16 fitting on this runtime is still to be confirmed.

Run the numbered cells in order. Cell 7 is the one-image inference run; Cell 8 verifies its PNG.

In [ ]:
# CELL 1 — Mount Drive, where the verified Phase 1 assets and logs are stored.
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# CELL 2 — Clone or update the pushed feature branch. No GitHub token is used.
from pathlib import Path
import subprocess

BRANCH = 'feat/baseline-runner'
REPO_URL = 'https://github.com/lemon4life/SC203.git'
REPO_DIR = Path('/content/SC203')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin'], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', '-B', BRANCH, f'origin/{BRANCH}'], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', BRANCH], check=True)

ASSETS_DIR = Path('/content/drive/MyDrive/lemmepass')
LOG_DIR = ASSETS_DIR / 'logs'
LOG_DIR.mkdir(parents=True, exist_ok=True)
commit = subprocess.run(['git', '-C', str(REPO_DIR), 'rev-parse', '--short', 'HEAD'], check=True, capture_output=True, text=True).stdout.strip()
print(f'Repository ready on {BRANCH} at {commit}')

In [ ]:
# CELL 3 — Confirm the actual Colab hardware and memory before running setup.
!nvidia-smi
!free -h
import platform, sys, torch
print('Python:', sys.version.split()[0])
print('Platform:', platform.platform())
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Select a GPU runtime before continuing.')
props = torch.cuda.get_device_properties(0)
free_bytes, total_bytes = torch.cuda.mem_get_info(0)
print(f'GPU: {props.name} | total={total_bytes / 2**30:.1f} GiB | free={free_bytes / 2**30:.1f} GiB')

In [ ]:
# CELL 4 — Repair the previously observed fresh-Colab Jedi gap, then check dependencies.
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'jedi==0.20.1'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'check'], check=True)

In [ ]:
# CELL 5 — Install/check DoSSR dependencies. This guarded script preserves Colab's torch and torchvision.
import sys
!{sys.executable} -u /content/SC203/scripts/colab_setup.py --requirements /content/SC203/external/DOSSR/requirements.txt --log-dir /content/drive/MyDrive/lemmepass/logs

In [ ]:
# CELL 6 — Stage exactly one RealSR image and create a dedicated FP16 smoke config/output folder.
import sys
!{sys.executable} -u /content/SC203/scripts/prepare_baseline_smoke.py --config /content/SC203/configs/baseline_realsr.yaml --smoke-dir /content/realsr_smoke_l4 --config-out /content/baseline_smoke_l4_fp16.yaml --output-root /content/outputs/baseline-smoke-l4-fp16 --log-dir /content/drive/MyDrive/lemmepass/logs

In [ ]:
%%bash
# CELL 7 — Run the one-image smoke test in FP16. Do not add CPU condition-encoder offload here.
cd /content/SC203
set -o pipefail
bash scripts/run_baseline.sh --config /content/baseline_smoke_l4_fp16.yaml --fp16 2>&1 | tee /content/baseline_smoke_l4_fp16_console.log
status=${PIPESTATUS[0]}
echo "SMOKE_RUNNER_EXIT_STATUS=$status"
exit "$status"

In [ ]:
# CELL 8 — Verify exactly one FP16 output. Stop here; the full baseline remains a separate run.
from pathlib import Path
from PIL import Image

output_dir = Path('/content/outputs/baseline-smoke-l4-fp16/seed_42')
images = sorted(output_dir.glob('*.png')) if output_dir.is_dir() else []
print('Output folder:', output_dir)
print('PNG count:', len(images))
for path in images:
    print(path)
if len(images) != 1:
    raise RuntimeError('Expected exactly one smoke-test PNG; do not start the full baseline.')
display(Image.open(images[0]))
print('ONE-IMAGE FP16 SMOKE TEST PASSED')

## If the smoke test fails

Do not run the 200-image baseline. Paste the last 50 lines of `/content/baseline_smoke_l4_fp16_console.log`, the full traceback, and the output of `!nvidia-smi`, `!free -h`, and `!cat /sys/fs/cgroup/memory.events`. The runner writes a timestamped log under `/content/drive/MyDrive/lemmepass/logs`.